In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pendulibrary.integrate import integrate_state
from pendulibrary.continuation import adaptive_cont
from pendulibrary.common_targetters import single_fixed
from pendulibrary.plotters import compare_fast, plot_timeline_grid
from pendulibrary.common import hamiltonian
from pendulibrary.utils import get_x0_corrected

int_tol = 1e-14

In [ ]:
Lr = 1.0
Mr = 1.0

# x0_ud, T_ud, tan_ud = get_x0_corrected(np.pi, 0.0, Lr, Mr, 5e-5)
# x0_du, T_du, tan_du = get_x0_corrected(0.0, np.pi, Lr, Mr, 5e-5)
x0s_dd, Ts_dd, tans_dd = get_x0_corrected(0.0, 0.0, Lr, Mr, 5e-5)

In [ ]:
Ts_dd

In [ ]:
cont_kwargs = dict(
    s0=5e-5, s_min=5e-5, S=20.0, tol=1e-10, max_iter=8, target_iter=5, rate=1.05
)

dd_ind = 1
targ = single_fixed(0, x0s_dd[dd_ind][0], 2, Lr, Mr, int_tol)
func = targ.g_dg_stm
X0, tan = targ.get_X(x0s_dd[dd_ind], Ts_dd[dd_ind]), tans_dd[dd_ind]

if np.dot(X0[1:-1], tan[1:-1]) < 0:
    tan *= -1
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    X0, func, tan, **cont_kwargs, exact_tangent=True
)

In [ ]:
x0s = np.array([targ.get_x0(X) for X in Xs])
periods = np.array([targ.get_period(X) for X in Xs])
hams = hamiltonian(x0s.T, Lr, Mr)

In [ ]:
np.savez_compressed(
    "../database/DDsp-P7a",
    x0s=x0s,
    periods=periods,
    eigs=eig_vals,
    tangents=np.column_stack((np.zeros_like(periods), tangents)),
    hamiltonians=hamiltonian(x0s.T, Lr, Mr),
    params=np.array([Lr, Mr]),
)

In [ ]:
tt, xx, ff = integrate_state(x0s[-1], periods[-1], Lr, Mr, 1e-14)

plot_timeline_grid(xx, tt, ff, Lr, 3, 5)